# Data Preprocessing

## Import Dependencies

In [211]:
import os
import warnings

import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.impute import KNNImputer
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split

from mappers import to_numeric

warnings.filterwarnings("ignore", category=DeprecationWarning)

## Data Loading

In [212]:
root = os.path.abspath(os.path.join(os.path.dirname(__name__), "..", "data"))
path = os.path.join(root, "raw.csv")

dataset = pd.read_csv(filepath_or_buffer=path)

## Data Cleaning

#### Remove unnecessary features
1. `Category URL`

2. `Service URL`

3. `Offer URL`

4. `Offer Name`

5. `Owner URL`

6. `Owner Name`


In [213]:
unnecessary_features = ["Category URL", "Service URL", "Offer URL", "Offer Name", "Owner URL", "Owner Name"]

dataset.drop(columns=unnecessary_features, inplace=True)

#### Convert text-based values to numeric values
1. Time: `Duration`, `Offer Response Time`, and `Owner Response Time`.

2. Percentage: `Owner Completion Rate`.

3. Boolean: `Owner Verified`.

4. Money: `Price`.


In [214]:
dataset["Owner Completion Rate"] = dataset["Owner Completion Rate"].mask(dataset["Owner Completion Rate"] == "لم يحسب بعد")

dataset["Owner Completion Rate"] = dataset["Owner Completion Rate"].replace("[\%,]", "", regex=True).astype(float)

dataset["Owner Verified"] = dataset["Owner Verified"].astype(int)

dataset["Price"] = dataset["Price"].replace("[\$,]", "", regex=True).astype(float)

dataset = to_numeric(dataset, columns=["Duration", "Offer Response Time", "Owner Response Time"])

## Encoding
1. `Owner Level`

2. `Category Name`

3. `Service Name`

#### Ordinal Encoding for `Owner Level` 

In [215]:
top_prices = (dataset.groupby("Owner Level", group_keys=False).apply(lambda x: x.nlargest(1, "Price")))

top_prices["Frequency"] = top_prices.apply(lambda row: dataset[(dataset["Owner Level"] == row["Owner Level"]) & (dataset["Price"] == row["Price"])].shape[0], axis=1)

order = (top_prices[["Owner Level", "Price", "Frequency"]].sort_values(by=["Price", "Frequency"])["Owner Level"].unique())

ordinal = OrdinalEncoder(categories=[order])

dataset["Owner Level"] = ordinal.fit_transform(dataset[["Owner Level"]])

#### One-Hot Encoding for `Category Name` and `Service Name`

In [216]:
categorical_features = ["Category Name", "Service Name"]

one_hot = OneHotEncoder()

encoded = one_hot.fit_transform(dataset[categorical_features])
encoded = pd.DataFrame(encoded.toarray(), columns=[col.split("_", 1)[-1] for col in one_hot.get_feature_names_out(categorical_features)])

dataset = pd.concat([dataset, encoded], axis=1).drop(categorical_features, axis=1)

## Missing Values Handling

#### Fill missing values using KNN imputation

In [217]:
imputer = KNNImputer(n_neighbors=3)

dataset = dataset.sample(frac=1).reset_index(drop=True)

dataset = pd.DataFrame(imputer.fit_transform(dataset), columns=dataset.columns)

## Outliers Handling

#### Detect and remove the outliers using Isolation Forest

In [218]:
numerical_data = dataset.select_dtypes(include=["number"])

isolation_forest = IsolationForest(contamination=0.01, random_state=42)

dataset["is_outlier"] = isolation_forest.fit_predict(numerical_data) == -1

In [219]:
dataset = dataset[dataset["is_outlier"] == False]

dataset.drop("is_outlier", axis=1, inplace=True)

dataset.shape

(7739, 303)

##  Split the dataset into *train* and *test*

#### Using stratified random splitting for representative data and fair sampling

In [220]:
train, test = train_test_split(dataset, test_size=0.05, random_state=42, stratify=dataset["Price"])

train, validation = train_test_split(train, test_size=0.06, random_state=42, stratify=train["Price"])

train.shape[0], test.shape[0], validation.shape[0]

(6910, 387, 442)

In [221]:
train["Price"].value_counts(normalize=True)

Price
5.0     0.556006
10.0    0.304776
25.0    0.048191
15.0    0.032127
20.0    0.027062
50.0    0.019826
30.0    0.006368
35.0    0.002315
40.0    0.001881
45.0    0.001447
Name: proportion, dtype: float64

In [222]:
test["Price"].value_counts(normalize=True)

Price
5.0     0.555556
10.0    0.304910
25.0    0.049096
15.0    0.031008
20.0    0.028424
50.0    0.020672
30.0    0.005168
40.0    0.002584
35.0    0.002584
Name: proportion, dtype: float64

In [223]:
validation["Price"].value_counts(normalize=True)

Price
5.0     0.556561
10.0    0.305430
25.0    0.047511
15.0    0.031674
20.0    0.027149
50.0    0.020362
30.0    0.006787
35.0    0.002262
40.0    0.002262
Name: proportion, dtype: float64

In [224]:
train_path = os.path.join(root, "train.csv")
test_path = os.path.join(root, "test.csv")
validation_path = os.path.join(root, "validation.csv")

train.to_csv(train_path, index=False)
test.to_csv(test_path, index=False)
validation.to_csv(validation_path, index=False)